# 🐚 Akoya Pearl Miner - Google Colab

**Setup:**
1. Runtime → Change runtime type → GPU (T4)
2. Jalankan semua cell di bawah
3. Miner akan jalan sampai session habis (~12 jam)

In [ ]:
# Cell 1: Check GPU
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv,noheader

In [ ]:
# Cell 2: Download Akoya Miner
!docker pull registry.akoyapool.com/akoya-miner:latest 2>/dev/null || true

# If docker not available, download binary directly
import os, subprocess, urllib.request, tarfile, shutil

MINER_DIR = "/content/akoya-miner"
os.makedirs(MINER_DIR, exist_ok=True)

# Extract from docker image
!apt-get update -qq && apt-get install -y -qq skopeo umoci jq > /dev/null 2>&1

print("Downloading Akoya miner image...")
!skopeo copy docker://registry.akoyapool.com/akoya-miner:latest oci:/content/akoya-oci:latest 2>/dev/null
!umoci unpack --image /content/akoya-oci:latest /content/akoya-bundle 2>/dev/null

# Copy miner files
!cp -r /content/akoya-bundle/rootfs/app/* {MINER_DIR}/
!chmod +x {MINER_DIR}/akoya-miner
!ls -la {MINER_DIR}/
print("✅ Miner ready!")

In [ ]:
# Cell 3: Setup GPU kernel symlink
import subprocess, os

MINER_DIR = "/content/akoya-miner"
lib_dir = f"{MINER_DIR}/lib"

cc = subprocess.run(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
    capture_output=True, text=True
).stdout.strip().split("\n")[0]

major, minor = cc.split(".")
print(f"GPU compute capability: {major}.{minor}")

if int(major) == 12: src = "blackwell"
elif int(major) == 9: src = "h100"
elif int(major) == 8 and int(minor) == 9: src = "ada"
elif int(major) == 7 and int(minor) == 5: src = "portable"  # T4
else: src = "portable"

target = f"{lib_dir}/libpearl_gemm_capi.so"
lib_file = f"{lib_dir}/libpearl_gemm_capi_{src}.so"

if os.path.lexists(target):
    os.unlink(target)

if os.path.exists(lib_file):
    os.symlink(lib_file, target)
    print(f"✅ Kernel: {src}")
else:
    # List available kernels
    available = [f for f in os.listdir(lib_dir) if 'gemm' in f]
    print(f"Available kernels: {available}")
    # Try portable
    portable = f"{lib_dir}/libpearl_gemm_capi_portable.so"
    if os.path.exists(portable):
        os.symlink(portable, target)
        print(f"✅ Using portable kernel")
    else:
        print(f"❌ No compatible kernel found")

In [ ]:
# Cell 4: START MINING 🚀
import subprocess, os, signal, time

MINER_DIR = "/content/akoya-miner"
WALLET = "prl1pq4v3a3677vymqt47jg6qvcxhgavrldmu4xe2k9hlx95lqqht2fyssutugf"
WORKER = "colab-t4"

os.environ["AKOYA_POOL_WALLET"] = WALLET
os.environ["AKOYA_POOL_WORKER"] = WORKER
os.environ["AKOYA_POOL_HOST"] = "pool-v2.akoyapool.com"
os.environ["AKOYA_POOL_PORT"] = "443"
os.environ["AKOYA_POOL_USE_TLS"] = "1"
os.environ["AKOYA_GPU_INDICES"] = "all"
os.environ["AKOYA_METRICS_PORT"] = "9100"
os.environ["AKOYA_PEARL_GEMM_LIB"] = f"{MINER_DIR}/lib/libpearl_gemm_capi.so"
os.environ["AKOYA_PEARL_MINING_LIB"] = f"{MINER_DIR}/lib/libpearl_mining_capi.so"
os.environ["LD_LIBRARY_PATH"] = f"{MINER_DIR}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

os.makedirs("/var/lib/akoya-miner", exist_ok=True)

print(f"🐚 Starting Akoya Pearl Miner")
print(f"   Wallet: {WALLET[:15]}...{WALLET[-8:]}")
print(f"   Worker: {WORKER}")
print(f"   Pool: pool-v2.akoyapool.com:443")
print("="*60)

proc = subprocess.Popen(
    [f"{MINER_DIR}/akoya-miner", "mine-blocks"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    while proc.poll() is None:
        line = proc.stdout.readline()
        if line:
            print(line.rstrip(), flush=True)
except KeyboardInterrupt:
    print("\n⏹️ Stopping miner...")
    proc.send_signal(signal.SIGINT)
    proc.wait(timeout=30)
    print("✅ Miner stopped.")